# TP1 — Classification FashionMNIST avec EfficientNet-B0

**Objectif :** classifier les images FashionMNIST en utilisant un **EfficientNet-B0** préentraîné sur ImageNet, une approche différente des architectures ResNet/VGG vues habituellement.

**Choix techniques :**
- Redimensionnement 64×64 et conversion niveaux de gris → RGB pour s'adapter à l'entrée d'un modèle ImageNet.
- Augmentation de données forte : `RandAugment` + `RandomErasing`.
- Transfer learning : tête de classification remplacée pour 10 classes.
- Régularisation : `label_smoothing`, `weight_decay`, dropout.
- Scheduler cosinus et early stopping pour stabiliser l'entraînement.

## 1. Imports et configuration

On fixe la graine aléatoire (`seed=42`) pour la reproductibilité, on sélectionne le device (GPU si disponible) et on affiche les versions des bibliothèques principales.

In [ ]:
# Imports des bibliothèques standard et scientifiques
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Imports PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Imports torchvision
import torchvision
from torchvision import datasets
from torchvision.transforms import v2  # API transforms v2 (plus moderne et composable)
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Outils d'évaluation
from sklearn.metrics import confusion_matrix, classification_report, f1_score

In [ ]:
# On fixe la graine aléatoire pour rendre les résultats reproductibles
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Sélection automatique du device : GPU CUDA si disponible, sinon CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Affichage des versions des bibliothèques et du device utilisé
print(f"PyTorch     : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Device      : {device}")

## 2. Chargement de FashionMNIST avec transforms v2

FashionMNIST est composé d'images 28×28 en niveaux de gris. Pour les rendre compatibles avec EfficientNet-B0 (préentraîné sur ImageNet en RGB), on les redimensionne en 64×64 et on duplique le canal de gris en 3 canaux RGB.

Deux pipelines distincts :
- **Train** : augmentation forte (`RandAugment`, `RandomErasing`) pour limiter le surapprentissage.
- **Val/Test** : uniquement le prétraitement déterministe (pas d'augmentation aléatoire).

In [ ]:
# Moyennes et écarts-types de normalisation ImageNet (sur 3 canaux RGB)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 64

# Pipeline d'entraînement : augmentation de données agressive
train_transform = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE)),                 # passage 28x28 -> 64x64
    v2.Grayscale(num_output_channels=3),             # gris -> RGB (3 canaux dupliqués)
    v2.RandAugment(num_ops=2, magnitude=9),          # 2 transformations aléatoires d'intensité 9
    v2.ToImage(),                                    # conversion en tv_tensor Image
    v2.ToDtype(torch.float32, scale=True),           # uint8 [0,255] -> float32 [0,1]
    v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),  # normalisation ImageNet
    v2.RandomErasing(p=0.25),                        # masquage aléatoire d'une zone (25% des cas)
])

# Pipeline validation/test : prétraitement déterministe sans augmentation
eval_transform = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.Grayscale(num_output_channels=3),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
# Téléchargement et chargement des jeux de données FashionMNIST
# train=True -> 60 000 images ; train=False -> 10 000 images de test
full_train = datasets.FashionMNIST(root="./data", train=True, download=True, transform=train_transform)
test_set = datasets.FashionMNIST(root="./data", train=False, download=True, transform=eval_transform)

# Découpage du train en train/validation (90% / 10%)
val_size = int(0.1 * len(full_train))
train_size = len(full_train) - val_size
generator = torch.Generator().manual_seed(SEED)  # split reproductible
train_set, val_subset = torch.utils.data.random_split(full_train, [train_size, val_size], generator=generator)

# Le subset de validation hérite du train_transform : on lui réassigne le transform d'évaluation.
# On recharge donc une vue séparée du dataset avec eval_transform pour les indices de validation.
val_base = datasets.FashionMNIST(root="./data", train=True, download=False, transform=eval_transform)
val_set = torch.utils.data.Subset(val_base, val_subset.indices)

print(f"Train : {len(train_set)} | Val : {len(val_set)} | Test : {len(test_set)}")

In [ ]:
# Noms des 10 classes FashionMNIST (dans l'ordre des labels 0 à 9)
CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

# Création des DataLoaders (lots de 128 images)
BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 3. Modèle EfficientNet-B0 préentraîné

On charge un EfficientNet-B0 avec ses poids ImageNet, puis on remplace la tête de classification (initialement 1000 classes) par une nouvelle tête : un `Dropout(0.3)` suivi d'une couche linéaire vers 10 classes.

In [ ]:
# Chargement d'EfficientNet-B0 avec les poids préentraînés ImageNet
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
model = efficientnet_b0(weights=weights)

# Récupération du nombre de features en entrée de la tête de classification
in_features = model.classifier[1].in_features

# Remplacement de la tête : Dropout(0.3) + couche linéaire vers 10 classes
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, 10),
)

# Déplacement du modèle sur le device choisi (GPU/CPU)
model = model.to(device)
print(model.classifier)

## 4. Entraînement

Configuration de l'entraînement (15 époques max) :
- **Optimiseur** : AdamW (`lr=1e-3`, `weight_decay=1e-4`).
- **Scheduler** : `CosineAnnealingLR` (décroissance cosinus du learning rate).
- **Loss** : `CrossEntropyLoss` avec `label_smoothing=0.1`.
- **Early stopping** : patience de 4 époques sur la `val_loss`.

In [ ]:
# Hyperparamètres d'entraînement
EPOCHS = 15
PATIENCE = 4  # early stopping : nombre d'époques sans amélioration toléré

# Fonction de perte avec lissage de label (régularisation)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Optimiseur AdamW : Adam avec découplage du weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Scheduler cosinus : le lr décroît selon une courbe cosinus sur EPOCHS époques
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
def run_epoch(loader, train_mode):
    """Exécute une époque (entraînement ou évaluation) et renvoie (loss moyenne, accuracy)."""
    # Bascule le modèle en mode train ou eval (affecte dropout / batchnorm)
    model.train() if train_mode else model.eval()

    running_loss, correct, total = 0.0, 0, 0
    # Désactive le calcul du gradient en évaluation pour économiser mémoire/temps
    context = torch.enable_grad() if train_mode else torch.no_grad()

    with context:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if train_mode:
                optimizer.zero_grad()  # remise à zéro des gradients

            outputs = model(images)            # passe avant
            loss = criterion(outputs, labels)  # calcul de la perte

            if train_mode:
                loss.backward()   # rétropropagation
                optimizer.step()  # mise à jour des poids

            # Cumul des métriques (pondérées par la taille du lot)
            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

In [ ]:
# Historique des métriques pour tracer les courbes ensuite
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

best_val_loss = float("inf")  # meilleure val_loss observée
epochs_no_improve = 0          # compteur pour l'early stopping
best_state = None              # sauvegarde des meilleurs poids

for epoch in range(1, EPOCHS + 1):
    # Une époque d'entraînement puis une de validation
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step()  # mise à jour du learning rate

    # Enregistrement de l'historique
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train_loss={train_loss:.4f} acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} acc={val_acc:.4f}")

    # Early stopping : on garde les meilleurs poids et on compte les époques sans amélioration
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping déclenché à l'époque {epoch} (patience={PATIENCE}).")
            break

# Restauration des meilleurs poids avant l'évaluation finale
if best_state is not None:
    model.load_state_dict(best_state)

## 5. Courbes de loss et d'accuracy

On visualise l'évolution de la perte et de l'accuracy sur le train et la validation pour détecter un éventuel surapprentissage.

In [ ]:
# Deux graphiques côte à côte : loss et accuracy
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbe de loss train vs val
axes[0].plot(epochs_range, history["train_loss"], marker="o", label="Train")
axes[0].plot(epochs_range, history["val_loss"], marker="o", label="Validation")
axes[0].set_title("Loss par époque")
axes[0].set_xlabel("Époque"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True)

# Courbe d'accuracy train vs val
axes[1].plot(epochs_range, history["train_acc"], marker="o", label="Train")
axes[1].plot(epochs_range, history["val_acc"], marker="o", label="Validation")
axes[1].set_title("Accuracy par époque")
axes[1].set_xlabel("Époque"); axes[1].set_ylabel("Accuracy"); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

## 6. Matrice de confusion sur le test set

On évalue le modèle sur les 10 000 images de test et on affiche la matrice de confusion pour identifier les confusions entre classes.

In [ ]:
# Collecte des prédictions et des labels réels sur l'ensemble de test
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

In [ ]:
# Calcul et affichage de la matrice de confusion avec seaborn
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Matrice de confusion - Test set")
plt.xlabel("Prédiction"); plt.ylabel("Réel")
plt.tight_layout()
plt.show()

## 7. Analyse des erreurs par classe

À partir de la matrice de confusion, on calcule l'accuracy par classe et on identifie les classes les plus difficiles ainsi que leurs confusions principales.

In [ ]:
# Accuracy par classe = diagonale / total des vrais exemples de la classe
per_class_acc = cm.diagonal() / cm.sum(axis=1)

print("Accuracy par classe (triée du plus faible au plus élevé) :\n")
for idx in np.argsort(per_class_acc):
    # Pour chaque classe, on cherche la classe avec laquelle elle est le plus confondue
    confusions = cm[idx].copy()
    confusions[idx] = 0  # on ignore les bonnes prédictions
    worst = confusions.argmax()
    print(f"{CLASS_NAMES[idx]:<12} : {per_class_acc[idx]:.3f} "
          f"(confondue surtout avec '{CLASS_NAMES[worst]}', {confusions[worst]} fois)")

## 8. Résultats finaux

Accuracy globale sur le test set et F1-score par classe (rapport de classification complet).

In [ ]:
# Accuracy globale sur le test set
test_acc = (all_preds == all_labels).mean()
print(f"Test accuracy globale : {test_acc:.4f}\n")

# F1-score macro (moyenne non pondérée des F1 par classe)
f1_macro = f1_score(all_labels, all_preds, average="macro")
print(f"F1-score macro : {f1_macro:.4f}\n")

# Rapport détaillé : précision, rappel et F1 par classe
print("Rapport de classification par classe :\n")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=3))